# 18  — Row Level Security (RLS)

## Step 1 — Inspect base Gold table

In [0]:
%sql

-- Group     | Streets visible
-- admin_group      | All streets (unrestricted)
-- safety_team      | All streets (needs full visibility for incident response)
-- public_dashboard_group | LOW danger tier only — HIGH/MEDIUM risk streets
--                    withheld from public-facing views without safety context
-- (others)         | No rows (deny by default)
--
-- Adapted from the reference project's brand-based RLS pattern (which
-- restricts by which car brand a group can see) to VStone's danger_score
-- tiers — the closest analogous "sensitive category" concept in this
-- dataset. Uses IS_MEMBER() with workspace-local groups (works on Free
-- Edition, unlike Unity Catalog account-level groups which need a
-- non-Free workspace).
SELECT * FROM vstone_catalog.gold.agg_top10_streets_by_pollution
ORDER BY avg_pollution DESC;

## Step 2 — Verify workspace groups

In [0]:
%sql
SHOW GROUPS;

## Step 3 — Confirm user identity and group membership

In [0]:
%sql
SELECT
  CURRENT_USER() AS current_user,
  IS_MEMBER('public_dashboard_group') AS is_public,
  IS_MEMBER('safety_team') AS is_safety,
  IS_MEMBER('admin_group') AS is_admin;


## Step 4 — RLS view on fact_street_readings

In [0]:
%sql

-- danger_score resolved via dim_street join (fact has no danger_score column)
CREATE OR REPLACE VIEW vstone_catalog.security.rls_fact_street_readings AS
SELECT
  f.street_id,
  d.street_name,
  f.reading_date,
  f.reading_ts,
  f.noise,
  f.pollution,
  f.raining_clipped,
  d.danger_score,
  f.gold_load_dt
FROM vstone_catalog.gold.fact_street_readings f
LEFT JOIN vstone_catalog.gold.dim_street d
  ON f.street_id = d.street_id AND d.__END_AT IS NULL
WHERE
  CASE
    WHEN IS_MEMBER('admin_group') THEN TRUE
    WHEN IS_MEMBER('safety_team') THEN TRUE
    WHEN IS_MEMBER('public_dashboard_group') THEN d.danger_score <= 0.34
    ELSE FALSE
  END;

##Step 5 — RLS view on agg_monthly_street_trend

In [0]:
%sql
CREATE OR REPLACE VIEW vstone_catalog.security.rls_monthly_street_trend AS
SELECT
  month_year, street_id, street_name, danger_score,
  reading_count, avg_noise, avg_pollution, avg_raining_pct, gold_load_dt
FROM vstone_catalog.gold.agg_monthly_street_trend
WHERE
  CASE
    WHEN IS_MEMBER('admin_group') THEN TRUE
    WHEN IS_MEMBER('safety_team') THEN TRUE
    WHEN IS_MEMBER('public_dashboard_group') THEN danger_score <= 0.34
    ELSE FALSE
  END;

## Step 6 — Verify filtered output (row count should shrink for a

In [0]:
%sql
-- public_dashboard_group member, stay full for admin_group/safety_team)
SELECT danger_score IS NOT NULL AS has_score, COUNT(*) AS row_count
FROM vstone_catalog.security.rls_fact_street_readings
GROUP BY danger_score IS NOT NULL;

-- Block direct table access, force traffic through the security views —
-- commented out by default, uncomment once the views are verified correct:
-- REVOKE SELECT ON TABLE vstone_catalog.gold.fact_street_readings FROM `account users`;
-- GRANT SELECT ON VIEW vstone_catalog.security.rls_fact_street_readings TO `account users`;
